# 02 · Silver Transformation Walkthrough
Trace how raw Bronze data becomes the Silver `orders` table.

In [ ]:
import os; os.chdir('..')
import duckdb
con = duckdb.connect('data/retailco_lakehouse.duckdb')

## Bronze → Silver: what changed?
- Joined products + customers dimensions
- Derived `gross_margin`, `is_completed`, `is_digital`
- Added `_silver_ts` for audit

In [ ]:
con.execute('''
    SELECT COUNT(*) AS bronze_rows FROM read_parquet('data/onelake/RetailCo.Lakehouse/Files/bronze/orders.parquet')
''').fetchone()

In [ ]:
con.execute('SELECT COUNT(*) AS silver_rows FROM silver.orders').fetchone()

In [ ]:
con.execute('''
    SELECT status, COUNT(*) AS cnt
    FROM read_parquet(''data/onelake/RetailCo.Lakehouse/Files/bronze/orders.parquet'')
    GROUP BY status ORDER BY cnt DESC
''').df()

## Margin analysis (new column in Silver)

In [ ]:
con.execute('''
    SELECT product_category,
           ROUND(AVG(gross_margin),2) AS avg_margin,
           ROUND(AVG(gross_margin/total_amount)*100,1) AS margin_pct
    FROM silver.orders
    WHERE is_completed
    GROUP BY product_category ORDER BY avg_margin DESC
''').df()